# Welcome to the Dask Tutorial

<img src="https://docs.dask.org/en/latest/_images/dask_horizontal.svg" align="right" width="30%" alt="Dask logo">

Dask is a parallel and distributed computing library that scales the existing Python and PyData ecosystem.

Dask can scale up to your full laptop capacity and out to a cloud cluster.

## An example Dask computation

In the following lines of code, we're reading the NYC taxi cab data from 2015 and finding the mean tip amount.

In [1]:
!pip install "dask[distributed]" --upgrade

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
    --------------------------------------- 0.0/1.0 MB 640.0 kB/s eta 0:00:02
   ------ --------------------------------- 0.2/1.0 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------  1.0/1.0 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 5.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 15.5 MB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 20.5 MB/s eta 0:00:01
   ---------------------------------------  1.5/1.5 MB 15.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 11.8 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2025.11.0
    Uninstalling dask-2025.11.0:
      Successfully uninstalled dask-2025.11.0
  Attempting uninstall:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-expr 1.1.0 requires dask==2024.5.0, but you have dask 2026.1.2 which is incompatible.


In [ ]:
import dask.dataframe as dd
from dask.distributed import Client

In [2]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 15.31 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57125,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:57146,Total threads: 4
Dashboard: http://127.0.0.1:57152/status,Memory: 3.83 GiB
Nanny: tcp://127.0.0.1:57128,


In [ ]:
ddf = dd.read_parquet(
    "yellow_tripdata_2023-01.parquet",
)

2026-03-11 10:14:06,835 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1100s; attempting restart: <WorkerState 'tcp://127.0.0.1:57146', name: 0, status: running, memory: 0, processing: 0>
2026-03-11 10:14:09,101 - distributed.nanny - WARNING - Restarting worker


In [9]:
ddf.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [10]:
#number of columns
ddf.shape[1]
#len(ddf.columns)


19

In [11]:
len(ddf.columns)

19

In [12]:
ddf.shape[0].compute()
#len(ddf)

3066766

In [13]:
len(ddf)

3066766

In [14]:
ddf = dd.read_parquet(
    "yellow_tripdata_2023-01.parquet",
    columns=["passenger_count", "tip_amount"],
    storage_options={"anon": True},
)

In [15]:
ddf.head()

,passenger_count,tip_amount
0,1.0,0.00
1,1.0,4.00
2,1.0,15.00
3,0.0,0.00
4,1.0,3.28


In [16]:
len(ddf.columns)

2

In [17]:
len(ddf)

3066766

In [18]:
unique_values_pc = ddf['passenger_count'].unique().compute()
unique_values_pc

2026-03-11 09:23:35,849 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 458e02225b30fb81639f6f25b3f08216 initialized by task ('shuffle-transfer-458e02225b30fb81639f6f25b3f08216', 0) executed on worker tcp://127.0.0.1:57147
2026-03-11 09:23:39,518 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 458e02225b30fb81639f6f25b3f08216 deactivated due to stimulus 'task-finished-1773217419.5174968'


0     1.0
1     0.0
2     4.0
3     2.0
4     3.0
5     5.0
6     6.0
7     8.0
8     7.0
9     9.0
10    NaN
Name: passenger_count, dtype: float64

In [19]:
unique_count = ddf['passenger_count'].nunique().compute()
unique_count


2026-03-11 09:23:48,378 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2c2064394e4d0583959eb4b7c8e8d785 initialized by task ('shuffle-transfer-2c2064394e4d0583959eb4b7c8e8d785', 0) executed on worker tcp://127.0.0.1:57147
2026-03-11 09:23:51,031 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2c2064394e4d0583959eb4b7c8e8d785 deactivated due to stimulus 'task-finished-1773217431.029497'


10

In [20]:
unique_value_count = ddf['passenger_count'].value_counts().compute()
unique_value_count

2026-03-11 09:24:02,334 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8d17dd7332cda5a82474512c208b00d0 initialized by task ('shuffle-transfer-8d17dd7332cda5a82474512c208b00d0', 0) executed on worker tcp://127.0.0.1:57147
2026-03-11 09:24:05,053 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8d17dd7332cda5a82474512c208b00d0 deactivated due to stimulus 'task-finished-1773217445.0527108'


passenger_count
0.0      51164
1.0    2261400
2.0     451536
3.0     106353
4.0      53745
5.0      42681
6.0      28124
7.0          6
8.0         13
9.0          1
Name: count, dtype: int64

In [21]:
unique_value_counts_sorted = ddf['passenger_count'].value_counts().compute().sort_values(ascending=False)
unique_value_counts_sorted

2026-03-11 09:24:14,335 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8d17dd7332cda5a82474512c208b00d0 initialized by task ('shuffle-transfer-8d17dd7332cda5a82474512c208b00d0', 0) executed on worker tcp://127.0.0.1:57147
2026-03-11 09:24:14,476 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8d17dd7332cda5a82474512c208b00d0 deactivated due to stimulus 'task-finished-1773217454.476477'


passenger_count
1.0    2261400
2.0     451536
3.0     106353
4.0      53745
0.0      51164
5.0      42681
6.0      28124
8.0         13
7.0          6
9.0          1
Name: count, dtype: int64

In [22]:
result = ddf.groupby("passenger_count").tip_amount.mean().compute()
result

passenger_count
0.0     2.870892
1.0     3.324993
2.0     3.589881
3.0     3.370632
4.0     3.281927
5.0     3.383753
6.0     3.355087
7.0     9.470000
8.0    13.389231
9.0     0.000000
NaN     3.733109
Name: tip_amount, dtype: float64

## What is [Dask]("https://www.dask.org/")?

There are many parts to the "Dask" the project:
* Collections/API also known as "core-library".
* Distributed -- to create clusters
* Intergrations and broader ecosystem


### Dask Collections

Dask provides **multi-core** and **distributed+parallel** execution on **larger-than-memory** datasets

We can think of Dask's APIs (also called collections)  at a high and a low level:

*  **High-level collections:**  Dask provides high-level Array, Bag, and DataFrame
   collections that mimic NumPy, lists, and pandas but can operate in parallel on
   datasets that don't fit into memory.
* **Low-level collections:**  Dask also provides low-level Delayed and Futures
   collections that give you finer control to build custom parallel and distributed computations.

### Dask Cluster

Most of the times when you are using Dask, you will be using a distributed scheduler, which exists in the context of a Dask cluster. The Dask cluster is structured as:


### Dask Ecosystem

In addition to the core Dask library and its distributed scheduler, the Dask ecosystem connects several additional initiatives, including:

- Dask-ML (parallel scikit-learn-style API)
- Dask-image
- Dask-cuDF
- Dask-sql
- Dask-snowflake
- Dask-mongo
- Dask-bigquery

Community libraries that have built-in dask integrations like:

- Xarray
- XGBoost
- Prefect
- Airflow

Dask deployment libraries
- Dask-kubernetes
- Dask-YARN
- Dask-gateway
- Dask-cloudprovider
- jobqueue

... When we talk about the Dask project we include all these efforts as part of the community.

## Dask Use Cases

Dask is used in multiple fields such as:

* Geospatial
* Finance
* Astrophysics
* Microbiology
* Environmental science

Check out the Dask [use cases](https://stories.dask.org/en/latest/) page that provides a number of sample workflows.

## Prepare

#### 1. You should clone this repository


    git clone http://github.com/dask/dask-tutorial

and then install necessary packages.
There are three different ways to achieve this, pick the one that best suits you, and ***only pick one option***.
They are, in order of preference:

#### 2a) Create a conda environment (preferred)

In the main repo directory


    conda env create -f binder/environment.yml
    conda activate dask-tutorial


#### 2b) Install into an existing environment

You will need the following core libraries


    conda install -c conda-forge ipycytoscape jupyterlab python-graphviz matplotlib zarr xarray pooch pyarrow s3fs scipy dask distributed dask-labextension

Note that these options will alter your existing environment, potentially changing the versions of packages you already
have installed.

## Useful Links

*  Reference
    *  [Docs](https://dask.org/)
    *  [Examples](https://examples.dask.org/)
    *  [Code](https://github.com/dask/dask/)
    *  [Blog](https://blog.dask.org/)
*  Ask for help
    *   [`dask`](http://stackoverflow.com/questions/tagged/dask) tag on Stack Overflow, for usage questions
    *   [github issues](https://github.com/dask/dask/issues/new) for bug reports and feature requests
    *   [discourse forum](https://dask.discourse.group/) for general, non-bug, questions and discussion
    *   Attend a live tutorial